<a href="https://colab.research.google.com/github/shehan16/Machine-Learning-and-AI/blob/main/2gis_Scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2GIS

In [ ]:
!pip install playwright playwright-stealth beautifulsoup4
!playwright install

!pip install requests-html


In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import csv
import math
import os
from google.colab import drive

In [ ]:
search_phrase = "Companies in al raafah umm al quwain"
base_url = "https://2gis.ae/search/"
search_url = f"{base_url}{search_phrase.replace(' ', '%20')}"

search_url

# $ Scrape From A Search Phrase  - Company name, Company Type, Location, Latitude , Longitude and Detailed URL $

## For Faster Execution - Doesn't go into detailed pages to extract all information.

#### Comapny name, Company Type, Location, Latitude , Longitude and Detailed URL

## Playwright Stealth

In [ ]:
search_phrase = "Holiday Homes"
base_url = "https://2gis.ae/search/"
search_url = f"{base_url}{search_phrase.replace(' ', '%20')}"

search_url

In [ ]:
import asyncio
import random
import csv
import os
import re
import base64
import json
from urllib.parse import urlparse, parse_qs
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

# Try to import Google Colab files module for automatic downloading
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# --- Configuration ---
HEADLESS_MODE = True

def extract_coords_from_url(url: str) -> tuple:
    """
    Extracts latitude and longitude from the 'stat' parameter of a 2GIS URL
    by robustly parsing it as JSON.
    """
    try:
        parsed_url = urlparse(url)
        stat_param = parse_qs(parsed_url.query).get('stat')

        if not stat_param:
            return None, None

        b64_string = stat_param[0]
        b64_string += '=' * (-len(b64_string) % 4)
        b64_bytes = b64_string.encode('utf-8')
        decoded_bytes = base64.urlsafe_b64decode(b64_bytes)
        decoded_string = decoded_bytes.decode('utf-8', errors='ignore')

        data = json.loads(decoded_string)

        geo_position = data.get('placeItem', {}).get('geoPosition', {})

        if 'lat' in geo_position and 'lon' in geo_position:
            latitude = geo_position['lat']
            longitude = geo_position['lon']
            return str(latitude), str(longitude)

    except (json.JSONDecodeError, KeyError, IndexError, TypeError):
        pass

    return None, None


def parse_page_with_beautifulsoup(html_content: str, page_number):
    """
    Parses the HTML content of a search results page and extracts company summary data.
    Handles both standard and alternative listing formats.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    selector = 'div._1kf6gff, div._1wgk7h51'
    companies = soup.select(selector)
    page_data = []

    print(f"--- Found {len(companies)} company summaries on this page ---")
    for company in companies:
        try:
            name, tag, link, location = "N/A", "N/A", "N/A", "N/A"

            # Standard selectors
            name_element = company.select_one('span._lvwrwt, span._9r89aog')
            tag_element = company.find('span', class_='_oqoid')
            link_element = company.find('a', class_='_1rehek')
            location_element = company.find('span', class_='_sfdp8cg')

            # Alternative selectors
            if not name_element:
                name_element = company.select_one('span._jkcb0i, span._hppqt6')
            if not tag_element:
                 tag_element = company.select_one('div._1idnaau span._oqoid')
            if not link_element:
                link_element = company.find('a')
            if not location_element:
                 location_element = company.select_one('div._klarpw')

            # Assign values
            if name_element:
                name = name_element.text.strip()
            if tag_element:
                tag = tag_element.text.strip()
            if link_element and link_element.get('href'):
                link = f"https://2gis.ae{link_element['href']}"
            if location_element:
                location = location_element.text.strip()

            company_details = {
                "Name": name,
                "Tag": tag,
                "Location (Summary)": location,
                "Link": link,
                "Page Number": page_number,
            }
            page_data.append(company_details)
        except Exception as e:
            print(f"Error extracting summary for one company: {e}")

    return page_data

def get_total_results_count(html_content: str) -> int:
    """
    Parses the initial HTML to find the total number of results.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    try:
        header = soup.find('h2', class_='_12jewu69')
        if header:
            count_span = header.find('span', class_='_1xhlznaa')
            if count_span:
                total_results = int(count_span.text.strip().replace(' ', ''))
                print(f"Found total results count: {total_results}")
                return total_results
    except (ValueError, AttributeError) as e:
        print(f"Could not parse total results count: {e}")

    print("Warning: Could not determine total number of results. Validation will be skipped.")
    return 0

async def scrape_2gis_summary_only(search_phrase: str, search_url: str):
    """
    Main async function to quickly scrape summary data only.
    It does NOT visit detail pages.
    """
    all_companies_data = []
    total_results_expected = 0

    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=HEADLESS_MODE)
        context = await browser.new_context()
        page = await context.new_page()

        print(f"Navigating to {search_url}...")
        await page.goto(search_url, timeout=90000, wait_until='domcontentloaded')

        try:
            await page.wait_for_selector('h2._12jewu69', timeout=20000)
            initial_html = await page.content()
            total_results_expected = get_total_results_count(initial_html)
        except Exception as e:
            print(f"Could not load initial page to get total results count. Error: {e}")
            await browser.close()
            return

        page_number = 1
        while True:
            print(f"\nScraping Search Results Page {page_number}... ✨")
            try:
                await page.wait_for_selector('div._1kf6gff', timeout=20000)
            except Exception as e:
                print("Could not find results container. Ending scrape.")
                break

            await asyncio.sleep(random.uniform(2, 4))
            html_content = await page.content()

            company_summaries = parse_page_with_beautifulsoup(html_content, str(page_number))

            for summary_data in company_summaries:
                link = summary_data.get("Link")
                latitude, longitude = None, None

                if link and link != "N/A":
                    latitude, longitude = extract_coords_from_url(link)

                summary_data["Latitude"] = latitude
                summary_data["Longitude"] = longitude

                print(f"-> Processing: {summary_data['Name']} (Lat: {latitude}, Lon: {longitude})")

                # Append summary data directly without visiting the detail page
                all_companies_data.append(summary_data)

            # --- Pagination Logic ---
            next_button_selector = 'div._n5hmn94:has(svg[style*="rotate(-90deg)"])'
            try:
                next_button = await page.query_selector(next_button_selector)
                if next_button:
                    print("Found 'Next Page' button. Clicking...")
                    await next_button.click()
                    await page.wait_for_load_state('domcontentloaded', timeout=15000)
                    page_number += 1
                else:
                    print("No 'Next Page' button found. Ending scrape.")
                    break
            except Exception as e:
                print(f"Could not click 'Next Page' or it's the last page. Error: {e}")
                break

        print("\nScraping finished.")
        await browser.close()

    # --- Save results to CSV ---
    if all_companies_data:
        # Use a different filename for summary-only results
        csv_filename = f"{search_phrase.replace(' ', '_')}_summary_only.csv"
        # Define the columns for the summary CSV
        column_order = [
            "Name", "Tag", "Location (Summary)", "Latitude", "Longitude",
            "Link", "Page Number"
        ]
        df = pd.DataFrame(all_companies_data)
        df = df.reindex(columns=column_order).fillna('')
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"\nSuccessfully saved {len(all_companies_data)} companies to '{csv_filename}'")

        if IS_COLAB:
            print("Attempting to download the file in Google Colab...")
            files.download(csv_filename)
    else:
        print("\nNo data was scraped. No CSV file generated.")

    # --- Validate results count and log failures ---
    extracted_count = len(all_companies_data)
    if total_results_expected > 0 and total_results_expected != extracted_count:
        print(f"\n⚠️ Validation FAILED: Expected ~{total_results_expected} results, but extracted {extracted_count}.")

        failed_csv = 'failed_urls.csv'
        file_exists = os.path.isfile(failed_csv)
        with open(failed_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(['failed_search_url'])
            writer.writerow([search_url])
        print(f"Logged the failed URL to '{failed_csv}'")
    elif total_results_expected > 0:
         print(f"\n✅ Validation PASSED: Extracted {extracted_count} results, which is close to the expected {total_results_expected}.")

In [ ]:
async def main():
    """
    Main function to set up and run the scraper.
    """
    #search_phrase = "Companies in al raafah umm al quwain"
    #search_phrase = "industries in bahya"
    search_phrase = "industries in bahya"
    base_url = "https://2gis.ae/search/"
    # Correctly format the URL by replacing spaces with %20
    search_url = f"{base_url}{search_phrase.replace(' ', '%20')}"

    await scrape_2gis_summary_only(search_phrase, search_url)

await main()

# $ Scrape From A Search Phrase - All Features $

Incorporating the Other Features for Extraction (Mobile Number, Website, PO Box)

In [ ]:
import asyncio
import random
import csv
import os
import re
import base64
import json
from urllib.parse import urlparse, parse_qs
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

# Try to import Google Colab files module for automatic downloading
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# --- Configuration ---
HEADLESS_MODE = True

def extract_coords_from_url(url: str) -> tuple:
    """
    Extracts latitude and longitude from the 'stat' parameter of a 2GIS URL
    by robustly parsing it as JSON.
    """
    try:
        parsed_url = urlparse(url)
        stat_param = parse_qs(parsed_url.query).get('stat')

        if not stat_param:
            return None, None

        b64_string = stat_param[0]
        b64_string += '=' * (-len(b64_string) % 4)
        b64_bytes = b64_string.encode('utf-8')
        decoded_bytes = base64.urlsafe_b64decode(b64_bytes)
        decoded_string = decoded_bytes.decode('utf-8', errors='ignore')

        data = json.loads(decoded_string)

        geo_position = data.get('placeItem', {}).get('geoPosition', {})

        if 'lat' in geo_position and 'lon' in geo_position:
            latitude = geo_position['lat']
            longitude = geo_position['lon']
            return str(latitude), str(longitude)

    except Exception:
        pass

    return None, None

def get_total_results_count(html_content: str) -> int:
    """
    Parses the initial HTML to find the total number of results.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    try:
        header = soup.find('h2', class_='_12jewu69')
        if header:
            count_span = header.find('span', class_='_1xhlznaa')
            if count_span:
                total_results = int(count_span.text.strip().replace(' ', ''))
                print(f"Found total results count: {total_results}")
                return total_results
    except (ValueError, AttributeError) as e:
        print(f"Could not parse total results count: {e}")

    print("Warning: Could not determine total number of results. Validation will be skipped.")
    return 0

def parse_page_with_beautifulsoup(html_content: str, page_number):
    """
    Parses the HTML content of a search results page and extracts company summary data.
    This version now adds a 'type' flag to differentiate between listing types.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    selector = 'div._1kf6gff, div._1wgk7h51'
    companies = soup.select(selector)
    page_data = []

    print(f"--- Found {len(companies)} company summaries on this page ---")
    for company in companies:
        try:
            listing_type = "alternative" # Default to alternative
            name, tag, link, location = "N/A", "N/A", "N/A", "N/A"

            # --- Try to parse as a Standard Company Listing first ---
            name_element = company.select_one('span._lvwrwt, span._9r89aog')

            if name_element:
                # It's a standard listing with a detailed business page
                listing_type = "standard"
                name = name_element.text.strip()
                tag_element = company.find('span', class_='_oqoid')
                tag = tag_element.text.strip() if tag_element else "N/A"
                link_element = company.find('a', class_='_1rehek')
                link = f"https://2gis.ae{link_element['href']}" if link_element and link_element.get('href') else "N/A"
                location_element = company.find('span', class_='_sfdp8cg')
                location = location_element.text.strip() if location_element else "N/A"
            else:
                # --- Fallback to Alternative Listing (e.g., a point of interest) ---
                alt_name_element = company.select_one('span._jkcb0i, span._hppqt6')
                if alt_name_element:
                    name = alt_name_element.get_text(strip=True)

                alt_tag_element = company.select_one('div._1idnaau span._oqoid')
                if alt_tag_element:
                    tag = alt_tag_element.get_text(strip=True)

                # UPDATED: Added location extraction for alternative listings
                alt_location_element = company.select_one('div._klarpw')
                if alt_location_element:
                    location = alt_location_element.get_text(strip=True)

                # For alternative listings, the link is on the main container
                link_element = company.find('a')
                link = f"https://2gis.ae{link_element['href']}" if link_element and link_element.get('href') else "N/A"


            company_details = {
                "Name": name,
                "Tag": tag,
                "Location (Summary)": location,
                "Link": link,
                "Page Number": page_number,
                "type": listing_type # Add the type flag
            }
            page_data.append(company_details)
        except Exception as e:
            print(f"Error extracting summary for one company: {e}")

    return page_data


async def scrape_company_details(page, url: str) -> dict:
    """
    Navigates to a specific company's page and scrapes address, phone, website, and P.O. Box.
    """
    details = {
        "Address": "",
        "Phone": "",
        "Website": "",
        "PO Box": "", # Added new field
    }
    try:
        print(f"    -> Navigating to detail page...")
        await page.goto(url, timeout=60000, wait_until='domcontentloaded')
        await page.wait_for_selector('div._1p8iqzw', timeout=20000)

        show_phone_button_selector = 'button._1tkj2hw'
        if await page.is_visible(show_phone_button_selector):
            print("    -> Clicking 'Show phone numbers' button...")
            await page.click(show_phone_button_selector)
            await page.wait_for_selector('div._49kxlr', timeout=10000)

        content = await page.content()
        soup = BeautifulSoup(content, 'html.parser')

        # 1. Extract Phone Numbers
        phone_elements = soup.select('a[href^="tel:"]')
        phone_numbers = [el.get_text(strip=True) for el in phone_elements if el.get_text(strip=True)]
        if phone_numbers:
            details["Phone"] = ', '.join(phone_numbers)

        # 2. Extract Website(s)
        websites = []
        website_elements = soup.select('div._49kxlr a._1rehek')
        for element in website_elements:
            website_text = element.get_text(strip=True)
            if '.' in website_text and website_text:
                websites.append(website_text)
        if websites:
            details["Website"] = ', '.join(sorted(list(set(websites))))

        # 3. Extract Address
        info_divs = soup.select('div._1p8iqzw')
        addresses = []
        for div in info_divs:
            text = ' '.join(div.text.strip().split())
            if '°' not in text:
                addresses.append(text)
        if addresses:
            details["Address"] = ', '.join(dict.fromkeys(addresses))

        # 4. Extract P.O. Box - NEW LOGIC
        contact_divs = soup.select('div._49kxlr')
        for div in contact_divs:
            text = div.get_text(strip=True)
            if text.lower().startswith('p.o.box'):
                details['PO Box'] = text
                break # Assume only one P.O. Box and stop searching

    except Exception as e:
        print(f"      Error scraping detail page: {e}")

    return details

async def scrape_2gis_working(search_phrase: str, search_url: str):
    """
    Main async function to scrape 2GIS. This version now extracts coordinates
    for all listing types that have a valid link.
    """
    all_companies_data = []
    total_results_expected = 0

    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=HEADLESS_MODE)
        context = await browser.new_context()
        page = await context.new_page()

        print(f"Navigating to {search_url}...")
        await page.goto(search_url, timeout=90000, wait_until='domcontentloaded')

        try:
            await page.wait_for_selector('h2._12jewu69', timeout=20000)
            initial_html = await page.content()
            total_results_expected = get_total_results_count(initial_html)
        except Exception as e:
            print(f"Could not load initial page to get total results count. Error: {e}")
            await browser.close()
            return

        page_number = 1
        while True:
            print(f"\nScraping Search Results Page {page_number}... ✨")
            try:
                await page.wait_for_selector('div._1kf6gff', timeout=20000)
            except Exception as e:
                print("Could not find results container. Ending scrape.")
                break

            await asyncio.sleep(random.uniform(2, 4))
            html_content = await page.content()

            company_summaries = parse_page_with_beautifulsoup(html_content, str(page_number))

            for summary_data in company_summaries:
                listing_type = summary_data.get("type")
                link = summary_data.get("Link")

                if listing_type == "standard":
                    if link and link != "N/A":
                        # Standard listings: Visit the detail page for full info
                        latitude, longitude = extract_coords_from_url(link)
                        summary_data["Latitude"] = latitude
                        summary_data["Longitude"] = longitude
                        print(f"-> Processing (Standard): {summary_data['Name']} (Lat: {latitude}, Lon: {longitude})")

                        detail_page = await context.new_page()
                        detailed_data = await scrape_company_details(detail_page, link)

                        full_company_data = {**summary_data, **detailed_data}
                        all_companies_data.append(full_company_data)

                        await detail_page.close()
                        await asyncio.sleep(random.uniform(1, 3))
                    else:
                        # Standard listing with no link, just append
                        all_companies_data.append(summary_data)
                else:
                    # Alternative listings: Extract coords from link but do NOT visit the page
                    latitude, longitude = None, None
                    if link and link != "N/A":
                        latitude, longitude = extract_coords_from_url(link)

                    summary_data["Latitude"] = latitude
                    summary_data["Longitude"] = longitude

                    print(f"-> Processing (Alternative Summary): {summary_data['Name']} (Lat: {latitude}, Lon: {longitude})")

                    summary_data['Address'] = summary_data.get('Location (Summary)', '')
                    all_companies_data.append(summary_data)

            # --- Pagination Logic ---
            next_button_selector = 'div._n5hmn94:has(svg[style*="rotate(-90deg)"])'
            try:
                next_button = await page.query_selector(next_button_selector)
                if next_button:
                    print("Found 'Next Page' button. Clicking...")
                    await next_button.click()
                    await page.wait_for_load_state('domcontentloaded', timeout=15000)
                    page_number += 1
                else:
                    print("No 'Next Page' button found. Ending scrape.")
                    break
            except Exception as e:
                print(f"Could not click 'Next Page' or it's the last page. Error: {e}")
                break

        print("\nScraping finished.")
        await browser.close()

    # --- Save results to CSV ---
    if all_companies_data:
        csv_filename = f"{search_phrase.replace(' ', '_')}_details.csv"
        column_order = [
            "Name", "Tag", "Address", "PO Box", "Phone", "Website",
            "Latitude", "Longitude", "Location (Summary)", "Link", "Page Number"
        ]
        df = pd.DataFrame(all_companies_data)
        df = df.reindex(columns=column_order).fillna('')
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"\nSuccessfully saved {len(all_companies_data)} companies to '{csv_filename}'")

        if IS_COLAB:
            print("Attempting to download the file in Google Colab...")
            files.download(csv_filename)
    else:
        print("\nNo data was scraped. No CSV file generated.")

    # --- Validate results count and log failures ---
    extracted_count = len(all_companies_data)
    if total_results_expected > 0 and total_results_expected != extracted_count:
        print(f"\n⚠️ Validation FAILED: Expected ~{total_results_expected} results, but extracted {extracted_count}.")

        failed_csv = 'failed_urls.csv'
        file_exists = os.path.isfile(failed_csv)
        with open(failed_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(['failed_search_url'])
            writer.writerow([search_url])
        print(f"Logged the failed URL to '{failed_csv}'")
    elif total_results_expected > 0:
         print(f"\n✅ Validation PASSED: Extracted {extracted_count} results, which is close to the expected {total_results_expected}.")

In [ ]:
async def main():
    """
    Main function to set up and run the scraper.
    """
    search_phrase = "malls in sharjah"

    base_url = "https://2gis.ae/search/"
    search_url = f"{base_url}{search_phrase.replace(' ', '%20')}"
    #search_url ='https://2gis.ae/dubai/search/Business%20workshops%20%2F%20seminars/rubricId/668?m=54.406%2C24.454829%2F11'

    await scrape_2gis_working(search_phrase, search_url)

# This is the standard way to run an asyncio program
await main()

# $ Scrape Categories - All Features $

### Identify all the Categories

In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright, TimeoutError
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
import asyncio
import json
import re
from urllib.parse import unquote

# This import is necessary for Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_URL = 'https://2gis.ae'
START_URL = f'{BASE_URL}/dubai/rubrics'

async def get_page_data(page, url, visited_urls, failed_urls_list):
    """
    Recursively scrapes category data based on a specific logic flow.
    This version decouples data extraction from navigation to prevent state errors.
    """
    if url in visited_urls:
       # print(f"  -> Skipping already visited URL: {url}")
        return None
    visited_urls.add(url)

    print(f"Scraping: {url}")
    try:
        await page.goto(url, wait_until='domcontentloaded', timeout=60000)
        await page.wait_for_timeout(3000)
    except TimeoutError:
        print(f"Timeout error navigating to {url}")
        failed_urls_list.append({'url': url, 'reason': 'Timeout on page load'})
        return None

    # --- Step 1 & 2: Extract all potential category links first ---
    child_links_to_visit = []

    # Extract Type 1 Categories ('_mq2eit')
    type1_locators = await page.locator('div._mq2eit a._591jgc').all()
    for link_locator in type1_locators:
        href = await link_locator.get_attribute('href')
        title = await link_locator.get_attribute('title')
        if href and title:
            child_links_to_visit.append({'url': f"{BASE_URL}{href}", 'name': title.strip()})

    # Extract Type 2 Categories ('_13w22bi')
    type2_locators = await page.locator('div._13w22bi').all()
    for rubric_div in type2_locators:
        try:
            link_locator = rubric_div.locator('a._1rehek')
            href = await link_locator.get_attribute('href')
            name = await link_locator.locator('span._1gh0934').inner_text()
            if href and name:
                 child_links_to_visit.append({'url': f"{BASE_URL}{href}", 'name': name.strip()})
        except Exception as e:
            print(f"  -> Warning: Could not parse a Type 2 category link on {url}. Error: {e}")

    # --- If we found links, recurse through them ---
    if child_links_to_visit:
        print(f"  -> Found {len(child_links_to_visit)} child categories. Now visiting them.")
        children_data = []
        for link_info in child_links_to_visit:
            child_result = await get_page_data(page, link_info['url'], visited_urls, failed_urls_list)
            if child_result:
                children_data.append({
                    "rubric": link_info['name'],
                    "url": link_info['url'],
                    "children": child_result
                })
        return children_data

    # --- Step 3: If no links were found, check for Stop Condition ---
    final_node_locator = page.locator('a._rdxuhv3:has-text("Places")')
    if await final_node_locator.count() > 0:
        try:
            count_text = await final_node_locator.first.locator('span._1xhlznaa').inner_text()
            category_name = ''
            try:
                category_name = await page.locator('h1._1lcm5n0').inner_text(timeout=5000)
            except TimeoutError:
                match = re.search(r'/search/([^/]+)/rubricId', url)
                if match:
                    category_name = unquote(match.group(1))
                else:
                    category_name = (await page.title()).split(' in Dubai')[0]

            print(f"  -> Final Node Found: {category_name.strip()} ({count_text.strip()} places)")
            return {
                "type": "final_node",
                "category_name": category_name.strip(),
                "count": int(re.sub(r'\D', '', count_text)),
                "url": url
            }
        except Exception as e:
            print(f"  -> Could not extract final node details from {url}: {e}")
            failed_urls_list.append({'url': url, 'reason': f'Error parsing final node: {e}'})
            return None

    # --- Step 4: If all above fails, log the URL ---
    print(f"  -> FAILURE: No categories or stop condition found on {url}")
    failed_urls_list.append({'url': url, 'reason': 'No recognized content'})
    return None


async def main():
    """
    Main function to initialize Playwright and start the scraping process.
    """
    failed_urls = []
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        page = await context.new_page()

        all_categories = await get_page_data(page, START_URL, set(), failed_urls)

        await browser.close()

    # --- Save files to the Colab environment ---

    # Save the successful results to a JSON file
    output_filename = '2gis_categories_treemap.json'
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(all_categories, f, ensure_ascii=False, indent=4)
    print(f"\nScraping complete. Main data saved to {output_filename}")

    # Save the failed URLs to a CSV file
    failed_filename = 'failed_urls.csv'
    if failed_urls:
        with open(failed_filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['url', 'reason'])
            writer.writeheader()
            writer.writerows(failed_urls)
        print(f"Found {len(failed_urls)} failed URLs. Details saved to {failed_filename}")

    # --- Trigger download in Google Colab ---
    if IN_COLAB:
        print("\nAttempting to download files to your local machine...")
        try:
            files.download(output_filename)
            if failed_urls:
                files.download(failed_filename)
            print("Download initiated successfully.")
        except Exception as e:
            print(f"Could not trigger download. You may need to download the files manually from the Colab file browser. Error: {e}")
    else:
        print("\nRun this script in a Google Colab notebook to download files automatically.")


In [ ]:
BASE_URL = 'https://2gis.ae'
START_URL = f'{BASE_URL}/dubai/rubrics/'

await main()

In [ ]:
import json
import pandas as pd
import os

def flatten_data_to_rows(node, path_prefix, all_rows):
    """
    Recursively traverses the nested JSON data and flattens it into a list of rows.

    Args:
        node: The current dictionary or list being processed from the JSON data.
        path_prefix: The current category path (e.g., "Eat out > Cafe").
        all_rows: A list to which the final flattened rows will be appended.
    """
    # Base Case: If the node is a final category with a count.
    if isinstance(node, dict) and node.get('type') == 'final_node':
        all_rows.append({
            'full_path': path_prefix,
            'final_category': node.get('category_name', 'N/A'),
            'company_count': node.get('count', 0),
            'url': node.get('url', 'N/A')
        })
        return

    # If the node is a dictionary with a 'children' key, process the children.
    if isinstance(node, dict) and 'children' in node:
        # The current rubric name is part of the path for its children.
        current_path = f"{path_prefix} > {node.get('rubric', '')}".strip(" >")
        # Recurse into the children
        flatten_data_to_rows(node['children'], current_path, all_rows)
        return

    # If the node is a list of categories/children, iterate through them.
    if isinstance(node, list):
        for item in node:
            # The path prefix remains the same for all items in the list.
            flatten_data_to_rows(item, path_prefix, all_rows)
        return


def view_category_data(json_filename='2gis_categories_treemap.json'):
    """
    Main function to load, display, and convert the category JSON file.
    """
    # --- Check if the file exists ---
    if not os.path.exists(json_filename):
        print(f"Error: The file '{json_filename}' was not found.")
        print("Please make sure the JSON file is in the same directory as this script.")
        # Create a dummy file for demonstration purposes if it doesn't exist
        print("Creating a dummy file for demonstration.")
        dummy_data = [
            {
                "rubric": "Eat out",
                "url": "...",
                "children": [
                    {
                        "rubric": "Cafe",
                        "url": "...",
                        "children": {
                            "type": "final_node",
                            "category_name": "Cafe",
                            "count": 500,
                            "url": "..."
                        }
                    },
                    {
                        "rubric": "Restaurants",
                        "url": "...",
                        "children": {
                            "type": "final_node",
                            "category_name": "Restaurants",
                            "count": 1200,
                            "url": "..."
                        }
                    }
                ]
            }
        ]
        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(dummy_data, f, indent=4)

    # --- Part 1: View the raw JSON file ---
    print("="*50)
    print(f"1. Viewing raw JSON from '{json_filename}'")
    print("="*50)

    with open(json_filename, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Pretty print the JSON data
    print(json.dumps(data, indent=4, ensure_ascii=False))

    # --- Part 2: Convert to a flat list (like a CSV) and display ---
    print("\n" + "="*50)
    print("2. Viewing the data as a flattened table (CSV format)")
    print("="*50)

    flattened_rows = []
    flatten_data_to_rows(data, "", flattened_rows)

    if not flattened_rows:
        print("Could not find any final category nodes to display in a table.")
        return

    # Use pandas to display the data in a nice tabular format
    df = pd.DataFrame(flattened_rows)

    # Set display options to show all columns and rows if needed
    pd.set_option('display.max_rows', 500)
    pd.set_option('display.max_columns', 50)
    pd.set_option('display.width', 150)

    print(df)

    # --- Part 3: Save the flattened data to a CSV file ---
    csv_filename = '2gis_categories_flattened.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"\nSuccessfully saved the flattened data to '{csv_filename}'")

    # If in Colab, offer to download the new CSV
    try:
        from google.colab import files
        print(f"\nAttempting to download '{csv_filename}'...")
        files.download(csv_filename)
    except ImportError:
        # Not in Colab, just print a message
        pass

In [ ]:
view_category_data()

In [ ]:
import pandas as pd
import os

def process_dataframe(csv_filename='2gis_categories_flattened.csv'):
    """
    Loads the flattened category data and processes the 'full_path' column
    to normalize the category hierarchy.
    """
    # --- Step 1: Check for the input file and load it ---
    if not os.path.exists(csv_filename):
        print(f"Error: The file '{csv_filename}' was not found.")
        print("Creating a dummy CSV file for demonstration purposes.")
        # Create a dummy dataframe that matches the user's example
        dummy_data = {
            'full_path': [
                "Eat out > Car service centers > Beauty > Entertainment > Health care > Car accessories > Groceries > Realty > Goods > Services > Tourism > Specialty stores > Sports > Education > Government > Repair, construction > Industrial goods > B2B services > Others > Religion > Christian cathedrals / churches",
                "Eat out > Car service centers > Beauty > Entertainment > Health care > Car accessories > Groceries > Realty > Goods > Services > Tourism > Specialty stores > Sports > Education > Government > Repair, construction > Industrial goods > B2B services > Others > Religion > Mosques",
                "Goods > Specialty stores > Bookstores",
                "Car service centers > Car wash"
            ],
            'final_category': [
                "Christian cathedrals / churches",
                "Mosques",
                "Bookstores",
                "Car wash"
            ],
            'company_count': [70, 150, 85, 200],
            'url': ['url1', 'url2', 'url3', 'url4']
        }
        df = pd.DataFrame(dummy_data)
        df.to_csv(csv_filename, index=False)

    df = pd.read_csv(csv_filename)
    print("--- Original DataFrame ---")
    print(df.head())

    # --- Step 2: Define the list of main categories ---
    # Note the use of .strip() to handle potential leading/trailing whitespace
    main_categories = [
        'Eat out', 'Car service centers', 'Beauty', 'Entertainment', 'Health care',
        'Car accessories', 'Groceries', 'Realty', 'Goods', 'Services', 'Tourism',
        'Specialty stores', 'Sports', 'Education', 'Government', 'Repair, construction',
        'Industrial goods', 'B2B services', 'Others'
    ]
    main_categories = [cat.strip() for cat in main_categories]

    # --- Step 3: Create a function to process each path ---
    def get_processed_path(path, categories):
        """
        Finds the last main category in the path and returns the substring
        from that point onwards.
        """
        if not isinstance(path, str):
            return ""

        last_pos = -1
        last_cat = None

        # Find the position of the last occurring main category
        for cat in categories:
            # Use rfind to find the last occurrence of the category string
            pos = path.rfind(cat)
            if pos > last_pos:
                last_pos = pos
                last_cat = cat

        if last_cat is not None:
            # Return the path starting from the last found main category
            return path[last_pos:]

        # If no main category was found in the path, return the original path
        return path

    # --- Step 4: Apply the function to the DataFrame ---
    df['processed_path'] = df['full_path'].apply(lambda x: get_processed_path(x, main_categories))

    # --- Step 5: Split the processed path into new columns ---
    # Split the 'processed_path' by ' > ' and expand into a new DataFrame
    path_split = df['processed_path'].str.split(' > ', expand=True)

    # Dynamically create column names like 'Main Category', 'Sub Category 1', etc.
    new_column_names = ['Main Category'] + [f'Sub Category {i}' for i in range(1, path_split.shape[1])]
    path_split.columns = new_column_names

    # --- Step 6: Combine the new columns with the original data ---
    # Drop the old 'full_path' and the intermediate 'processed_path'
    df_processed = df.drop(columns=['full_path', 'processed_path'])
    final_df = pd.concat([path_split, df_processed], axis=1)

    # Reorder columns to have the category hierarchy first
    original_cols = ['final_category', 'company_count', 'url']
    final_df = final_df[new_column_names + original_cols]

    # print("\n--- Processed DataFrame ---")
    # print(final_df)

    # --- Step 7: Save the result to a new CSV file ---
    output_filename = '2gis_categories_processed.csv'
    final_df.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"\nSuccessfully saved the processed data to '{output_filename}'")

    # If in Colab, offer to download the new CSV
    try:
        from google.colab import files
        print(f"\nAttempting to download '{output_filename}'...")
        files.download(output_filename)
    except ImportError:
        # Not in Colab, just print a message
        pass


In [ ]:
process_dataframe()

Loading Categories.csv from Google Drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
FILE_LOCATION = '/content/drive/MyDrive/UAE Company Scraper/2GIS/2gis_categories_processed.csv'
categories = pd.read_csv(FILE_LOCATION)
categories.tail()

## Run for all the Categories and URLs in the Categories Df

In [ ]:
import asyncio
import random
import csv
import os
import re
import base64
import json
from urllib.parse import urlparse, parse_qs
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

# Try to import Google Colab files module for automatic downloading
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# --- Configuration ---
HEADLESS_MODE = True

def extract_coords_from_url(url: str) -> tuple:
    """
    Extracts latitude and longitude from the 'stat' parameter of a 2GIS URL
    by robustly parsing it as JSON.
    """
    try:
        parsed_url = urlparse(url)
        stat_param = parse_qs(parsed_url.query).get('stat')

        if not stat_param:
            return None, None

        b64_string = stat_param[0]
        b64_string += '=' * (-len(b64_string) % 4)
        b64_bytes = b64_string.encode('utf-8')
        decoded_bytes = base64.urlsafe_b64decode(b64_bytes)
        decoded_string = decoded_bytes.decode('utf-8', errors='ignore')

        data = json.loads(decoded_string)

        geo_position = data.get('placeItem', {}).get('geoPosition', {})

        if 'lat' in geo_position and 'lon' in geo_position:
            latitude = geo_position['lat']
            longitude = geo_position['lon']
            return str(latitude), str(longitude)

    except Exception:
        pass

    return None, None

def get_total_results_count(html_content: str) -> int:
    """
    Parses the initial HTML to find the total number of results.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    try:
        header = soup.find('h2', class_='_12jewu69')
        if header:
            count_span = header.find('span', class_='_1xhlznaa')
            if count_span:
                total_results = int(count_span.text.strip().replace(' ', ''))
                print(f"Found total results count: {total_results}")
                return total_results
    except (ValueError, AttributeError) as e:
        print(f"Could not parse total results count: {e}")

    print("Warning: Could not determine total number of results. Validation will be skipped.")
    return 0

def parse_page_with_beautifulsoup(html_content: str, page_number):
    """
    Parses the HTML content of a search results page and extracts company summary data.
    This version now adds a 'type' flag to differentiate between listing types.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    selector = 'div._1kf6gff, div._1wgk7h51'
    companies = soup.select(selector)
    page_data = []

    print(f"--- Found {len(companies)} company summaries on this page ---")
    for company in companies:
        try:
            listing_type = "alternative" # Default to alternative
            name, tag, link, location = "N/A", "N/A", "N/A", "N/A"

            # --- Try to parse as a Standard Company Listing first ---
            name_element = company.select_one('span._lvwrwt, span._9r89aog')

            if name_element:
                # It's a standard listing with a detailed business page
                listing_type = "standard"
                name = name_element.text.strip()
                tag_element = company.find('span', class_='_oqoid')
                tag = tag_element.text.strip() if tag_element else "N/A"
                link_element = company.find('a', class_='_1rehek')
                link = f"https://2gis.ae{link_element['href']}" if link_element and link_element.get('href') else "N/A"
                location_element = company.find('span', class_='_sfdp8cg')
                location = location_element.text.strip() if location_element else "N/A"
            else:
                # --- Fallback to Alternative Listing (e.g., a point of interest) ---
                alt_name_element = company.select_one('span._jkcb0i, span._hppqt6')
                if alt_name_element:
                    name = alt_name_element.get_text(strip=True)

                alt_tag_element = company.select_one('div._1idnaau span._oqoid')
                if alt_tag_element:
                    tag = alt_tag_element.get_text(strip=True)

                # UPDATED: Added location extraction for alternative listings
                alt_location_element = company.select_one('div._klarpw')
                if alt_location_element:
                    location = alt_location_element.get_text(strip=True)

                # For alternative listings, the link is on the main container
                link_element = company.find('a')
                link = f"https://2gis.ae{link_element['href']}" if link_element and link_element.get('href') else "N/A"


            company_details = {
                "Name": name,
                "Tag": tag,
                "Location (Summary)": location,
                "Link": link,
                "Page Number": page_number,
                "type": listing_type # Add the type flag
            }
            page_data.append(company_details)
        except Exception as e:
            print(f"Error extracting summary for one company: {e}")

    return page_data


async def scrape_company_details(page, url: str) -> dict:
    """
    Navigates to a specific company's page and scrapes address, phone, website, and P.O. Box.
    """
    details = {
        "Address": "",
        "Phone": "",
        "Website": "",
        "PO Box": "", # Added new field
    }
    try:
        print(f"    -> Navigating to detail page...")
        await page.goto(url, timeout=60000, wait_until='domcontentloaded')
        await page.wait_for_selector('div._1p8iqzw', timeout=20000)

        show_phone_button_selector = 'button._1tkj2hw'
        if await page.is_visible(show_phone_button_selector):
            print("    -> Clicking 'Show phone numbers' button...")
            await page.click(show_phone_button_selector)
            await page.wait_for_selector('div._49kxlr', timeout=10000)

        content = await page.content()
        soup = BeautifulSoup(content, 'html.parser')

        # 1. Extract Phone Numbers
        phone_elements = soup.select('a[href^="tel:"]')
        phone_numbers = [el.get_text(strip=True) for el in phone_elements if el.get_text(strip=True)]
        if phone_numbers:
            details["Phone"] = ', '.join(phone_numbers)

        # 2. Extract Website(s)
        websites = []
        website_elements = soup.select('div._49kxlr a._1rehek')
        for element in website_elements:
            website_text = element.get_text(strip=True)
            if '.' in website_text and website_text:
                websites.append(website_text)
        if websites:
            details["Website"] = ', '.join(sorted(list(set(websites))))

        # 3. Extract Address
        info_divs = soup.select('div._1p8iqzw')
        addresses = []
        for div in info_divs:
            text = ' '.join(div.text.strip().split())
            if '°' not in text:
                addresses.append(text)
        if addresses:
            details["Address"] = ', '.join(dict.fromkeys(addresses))

        # 4. Extract P.O. Box - NEW LOGIC
        contact_divs = soup.select('div._49kxlr')
        for div in contact_divs:
            text = div.get_text(strip=True)
            if text.lower().startswith('p.o.box'):
                details['PO Box'] = text
                break # Assume only one P.O. Box and stop searching

    except Exception as e:
        print(f"      Error scraping detail page: {e}")

    return details

async def scrape_2gis_working(search_phrase: str, search_url: str):
    """
    Main async function to scrape 2GIS. This version now extracts coordinates
    for all listing types that have a valid link.
    """
    all_companies_data = []
    total_results_expected = 0

    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=HEADLESS_MODE)
        context = await browser.new_context()
        page = await context.new_page()

        print(f"Navigating to {search_url}...")
        await page.goto(search_url, timeout=90000, wait_until='domcontentloaded')

        try:
            await page.wait_for_selector('h2._12jewu69', timeout=20000)
            initial_html = await page.content()
            total_results_expected = get_total_results_count(initial_html)
        except Exception as e:
            print(f"Could not load initial page to get total results count. Error: {e}")
            await browser.close()
            return

        page_number = 1
        while True:
            print(f"\nScraping Search Results Page {page_number}... ✨")
            try:
                await page.wait_for_selector('div._1kf6gff', timeout=20000)
            except Exception as e:
                print("Could not find results container. Ending scrape.")
                break

            await asyncio.sleep(random.uniform(2, 4))
            html_content = await page.content()

            company_summaries = parse_page_with_beautifulsoup(html_content, str(page_number))

            for summary_data in company_summaries:
                listing_type = summary_data.get("type")
                link = summary_data.get("Link")

                if listing_type == "standard":
                    if link and link != "N/A":
                        # Standard listings: Visit the detail page for full info
                        latitude, longitude = extract_coords_from_url(link)
                        summary_data["Latitude"] = latitude
                        summary_data["Longitude"] = longitude
                        print(f"-> Processing (Standard): {summary_data['Name']} (Lat: {latitude}, Lon: {longitude})")

                        detail_page = await context.new_page()
                        detailed_data = await scrape_company_details(detail_page, link)

                        full_company_data = {**summary_data, **detailed_data}
                        all_companies_data.append(full_company_data)

                        await detail_page.close()
                        await asyncio.sleep(random.uniform(1, 3))
                    else:
                        # Standard listing with no link, just append
                        all_companies_data.append(summary_data)
                else:
                    # Alternative listings: Extract coords from link but do NOT visit the page
                    latitude, longitude = None, None
                    if link and link != "N/A":
                        latitude, longitude = extract_coords_from_url(link)

                    summary_data["Latitude"] = latitude
                    summary_data["Longitude"] = longitude

                    print(f"-> Processing (Alternative Summary): {summary_data['Name']} (Lat: {latitude}, Lon: {longitude})")

                    summary_data['Address'] = summary_data.get('Location (Summary)', '')
                    all_companies_data.append(summary_data)

            # --- Pagination Logic ---
            next_button_selector = 'div._n5hmn94:has(svg[style*="rotate(-90deg)"])'
            try:
                next_button = await page.query_selector(next_button_selector)
                if next_button:
                    print("Found 'Next Page' button. Clicking...")
                    await next_button.click()
                    await page.wait_for_load_state('domcontentloaded', timeout=15000)
                    page_number += 1
                else:
                    print("No 'Next Page' button found. Ending scrape.")
                    break
            except Exception as e:
                print(f"Could not click 'Next Page' or it's the last page. Error: {e}")
                break

        print("\nScraping finished.")
        await browser.close()

    # --- Save results to CSV ---
    if all_companies_data:
        csv_filename = f"{search_phrase.replace(' ', '_')}_details.csv"
        column_order = [
            "Name", "Tag", "Address", "PO Box", "Phone", "Website",
            "Latitude", "Longitude", "Location (Summary)", "Link", "Page Number"
        ]
        df = pd.DataFrame(all_companies_data)
        df = df.reindex(columns=column_order).fillna('')
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"\nSuccessfully saved {len(all_companies_data)} companies to '{csv_filename}'")

        if IS_COLAB:
            print("Attempting to download the file in Google Colab...")
            files.download(csv_filename)
    else:
        print("\nNo data was scraped. No CSV file generated.")

    # --- Validate results count and log failures ---
    extracted_count = len(all_companies_data)
    if total_results_expected > 0 and total_results_expected != extracted_count:
        print(f"\n⚠️ Validation FAILED: Expected ~{total_results_expected} results, but extracted {extracted_count}.")

        failed_csv = 'failed_urls.csv'
        file_exists = os.path.isfile(failed_csv)
        with open(failed_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(['failed_search_url'])
            writer.writerow([search_url])
        print(f"Logged the failed URL to '{failed_csv}'")
    elif total_results_expected > 0:
         print(f"\n✅ Validation PASSED: Extracted {extracted_count} results, which is close to the expected {total_results_expected}.")

In [ ]:
async def main():
    """
    Main function to set up and run the scraper.
    """
    # Assign the dataframe with the Columns "final_category" and "url" which you need to extract
    categories_dataframe = test_df

    # Iterate through each row and run the function
    for index, row in categories_dataframe.iterrows():
        search_phrase = row['final_category']
        search_url = row['url']
        await scrape_2gis_working(search_phrase, search_url)

# This is the standard way to run an asyncio program
await main()

In [ ]:
categories[4:5]

In [ ]:
categories[1107:1108]

In [ ]:
test_df = pd.concat([categories[4:5], categories[1107:1108]])
test_df

Testing Longitude and Latitude

In [ ]:
url1 = 'https://2gis.ae/dubai/firm/70000001006745400?stat=eyJwYXJlbnRUYWJJZCI6IjJmNTBlNWVkLWU3MzMtNDYwNS1hY2MzLTQ3MDFjOGQ1YzNiMiIsImZvcmtFdmVudE9yZGluYWwiOjIsInVpRWxlbWVudCI6eyJuYW1lIjoicGxhY2VDYXJkTWluaSIsIm93bmVyTmFtZSI6InNlYXJjaFJlc3VsdHNMaXN0IiwicG9zaXRpb24iOjN9LCJwbGFjZUl0ZW0iOnsiYWRzSWQiOiIxMjIzNDM2Mzc5NDUxNTU2MDk2IiwiZW50aXR5Ijp7ImlkIjoiNzAwMDAwMDEwMDY3NDU0MDAiLCJ0eXBlIjoiYnJhbmNoIiwic2VnbWVudEluZm8iOnsiYmFzZUxvY2FsZSI6ImVuX0FFIiwic2VnbWVudElkIjoiOTkifX0sImdlb1Bvc2l0aW9uIjp7ImxvbiI6NTUuMTQyMSwibGF0IjoyNS4wNjc1ODJ9LCJoeWJyaWRFbnRpdGllcyI6W3siaWQiOiIxMzkzMzY0NzAwMjU2MjE0MCIsInR5cGUiOiJidWlsZGluZyJ9XSwiYWRzU2VhcmNoIjpmYWxzZSwibWFpblJ1YnJpYyI6IjI2OSIsImlzRGVsZXRlZCI6ZmFsc2UsIm9yZyI6IjcwMDAwMDAxMDA2MzcwNzg3IiwiaGFzTW9kZWwiOmZhbHNlLCJyZXN1bHRDbGFzcyI6MSwicG9zaXRpb24iOjN9LCJzZWFyY2hSZXN1bHRzTGlzdCI6eyJzaXplIjo1LCJzZWFyY2hIYXNoIjoiOTNlMjY0ZTAtZmZiMS00OWJmLWE2ODktZjdhMjJkYjg3ZjQ4IiwidHlwb0NvcnJlY3RlZCI6dHJ1ZSwic2VhcmNoVHlwZSI6MSwiZmlyc3RSZXN1bHRDbGFzcyI6MSwiaGFzR2VvUmVzdHJpY3Rpb24iOmZhbHNlLCJwYXJ0aWFsUmVzdWx0IjpmYWxzZSwib3V0dmlld3BvcnQiOlt7ImxhdCI6MjUuNDE0OTgsImxvbiI6NTQuOTQ3ODMxfSx7ImxhdCI6MjUuMDE0NTg2LCJsb24iOjU0Ljk0NzgzMX0seyJsYXQiOjI1LjAxNDU4NiwibG9uIjo1NS41NDEwOTN9LHsibGF0IjoyNS40MTQ5OCwibG9uIjo1NS41NDEwOTN9XSwic2VhcmNoU2VnbWVudElkIjoiOTkiLCJydHdWZXJzaW9uIjoiIn19'
#decode_url_for_coords(url1)
extract_coords_from_url(url1)
